# 🎬 OpenShorts on Google Colab
Generator video pendek 9:16 otomatis dari video panjang / YouTube menggunakan AI.

**Fitur yang sudah aktif:**
- Custom LLM: `Gemini 3.8 Flash` via `https://my.ivyy.dpdns.org/v1`
- Whisper ASR untuk transkrip otomatis
- YOLOv8 + MediaPipe auto reframing
- Auto subtitle & hook
- Cloudflare Tunnel untuk akses web UI publik langsung dari browser Anda.

### Langkah 1: Cek GPU

In [ ]:
!nvidia-smi

### Langkah 2: Clone Repo & Install Dependensi Sistem
*(Lewati git clone jika notebook ini dijalankan langsung di dalam folder repo)*

In [ ]:
import os
if not os.path.exists('app.py'):
    !git clone https://github.com/digitaldeveloperinfo/shrotclone.git /content/openshorts
    %cd /content/openshorts

!apt-get update -qq
!apt-get install -y -qq ffmpeg curl nodejs npm

### Langkah 3: Install Dependensi Python & Frontend

In [ ]:
# Install Python requirements
!pip install -r requirements.txt
!pip install "mediapipe==0.10.14" "scenedetect>=0.6.4" py3langid faster-whisper ultralytics torch torchvision transnetv2-pytorch json-repair

# Install Dashboard dependencies
!cd dashboard && npm ci

### Langkah 4: Konfigurasi Environment (.env)
Secara default menggunakan endpoint Custom LLM Gemini 3.8 Flash yang sudah dikonfigurasi.

In [ ]:
env_content = '''
LLM_BASE_URL=https://my.ivyy.dpdns.org/v1
LLM_API_KEY=sk-5f238e76072d7926-8d3e86-14b7d8e3
LLM_MODEL=antigravity/gemini-3.8-flash-tiered
LLM_PROVIDER=openai
LLM_TIMEOUT=120
LLM_SCORE_BATCH=3
QUALITY_GATE_MIN_HEIGHT=0
RATE_LIMIT_ENABLED=0
DEBUG_LOGS=true
'''
with open('.env', 'w') as f:
    f.write(env_content.strip() + '\n')
print('✅ File .env berhasil dibuat!')

### Langkah 5: Jalankan OpenShorts & Dapatkan Public URL
Sel ini akan menjalankan Backend (FastAPI), Frontend (Vite), dan Cloudflare Tunnel.
Tunggu beberapa detik hingga URL **https://xxxx.trycloudflare.com** muncul di output di bawah.

In [ ]:
import os, subprocess, time, urllib.request, re

# 1. Download Cloudflared
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('⬇️ Mengunduh Cloudflared tunnel...')
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '/usr/local/bin/cloudflared'
    )
    os.chmod('/usr/local/bin/cloudflared', 0o755)

# 2. Jalankan Backend (FastAPI) di background
print('🚀 Menjalankan Backend API (:8000)...')
backend_proc = subprocess.Popen(
    ['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

# 3. Jalankan Frontend (Vite) di background
print('⚡ Menjalankan Frontend Vite (:5173)...')
frontend_proc = subprocess.Popen(
    ['npm', 'run', 'dev', '--', '--host', '0.0.0.0', '--port', '5173'],
    cwd='dashboard',
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

time.sleep(5)

# 4. Jalankan Cloudflare Tunnel ke port 5173
print('🌐 Membuka Cloudflare Tunnel publik...')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:5173'],
    stderr=subprocess.PIPE,
    stdout=subprocess.PIPE,
    text=True
)

for line in tunnel_proc.stderr:
    match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        print('\n' + '='*65)
        print(f'🎉 OPENSHORTS BERHASIL AKTIF!')
        print(f'👉 Akses Dashboard di URL ini: {url}')
        print('='*65 + '\n')
        break

# Jaga agar sel tetap berjalan
try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    print('Mematikan server...')
    tunnel_proc.terminate()
    backend_proc.terminate()
    frontend_proc.terminate()
